# Gold: hospital_daily_summary

One row per hospital per calendar day, built from `silver.lab_results` per the STTM (GOLD/sttm_silver_to_gold.xlsx). Full rebuild with day-over-day trend via LAG().

In [0]:
# Load type: FULL rebuild (CREATE OR REPLACE TABLE ... AS SELECT). LEFT JOINs from the aggregated lab_results driver.
catalog = "caresyncsydatabricks"

spark.sql(f"""
CREATE OR REPLACE TABLE {catalog}.gold.hospital_daily_summary AS
WITH daily AS (
    SELECT
        hospital_id,
        test_date                               AS summary_date,
        COUNT(*)                                AS total_tests,
        COUNT(DISTINCT patient_id)              AS unique_patients_tested,
        COUNT(DISTINCT doctor_id)               AS active_doctor_count,
        COUNT_IF(result_status = 'Abnormal')    AS abnormal_test_count,
        COUNT_IF(result_status = 'Critical')    AS critical_test_count
    FROM {catalog}.silver.lab_results
    GROUP BY hospital_id, test_date
),
with_lag AS (
    SELECT
        d.*,
        LAG(total_tests) OVER (PARTITION BY hospital_id ORDER BY summary_date) AS prev_day_total_tests
    FROM daily d
)
SELECT
    CAST(w.hospital_id AS INT)                                              AS hospital_id,
    h.hospital_name,
    w.summary_date,
    c.day_name,
    c.is_weekend,
    CAST(w.total_tests AS INT)                                              AS total_tests,
    CAST(w.unique_patients_tested AS INT)                                   AS unique_patients_tested,
    CAST(w.active_doctor_count AS INT)                                      AS active_doctor_count,
    CAST(w.abnormal_test_count AS INT)                                      AS abnormal_test_count,
    CAST(w.critical_test_count AS INT)                                      AS critical_test_count,
    CAST(w.abnormal_test_count / w.total_tests * 100 AS DECIMAL(5,2))       AS abnormal_rate_pct,
    CAST(w.critical_test_count / w.total_tests * 100 AS DECIMAL(5,2))       AS critical_rate_pct,
    CAST(w.prev_day_total_tests AS INT)                                     AS prev_day_total_tests,
    CASE WHEN w.prev_day_total_tests IS NULL OR w.prev_day_total_tests = 0 THEN NULL
         ELSE CAST((w.total_tests - w.prev_day_total_tests) / w.prev_day_total_tests * 100 AS DECIMAL(6,2))
    END                                                                     AS test_volume_change_pct
FROM with_lag w
LEFT JOIN {catalog}.silver.hospitals h
       ON w.hospital_id = h.hospital_id
LEFT JOIN {catalog}.silver.calendar c
       ON c.date_key = CAST(date_format(w.summary_date, 'yyyyMMdd') AS INT)
""")

display(spark.table(f"{catalog}.gold.hospital_daily_summary").orderBy("hospital_id", "summary_date").limit(10))

In [0]:
# Validation: grain uniqueness (hospital_id, summary_date) and test count reconciliation with silver.lab_results
display(spark.sql(f"""
SELECT
    COUNT(*)                                                    AS gold_rows,
    COUNT(DISTINCT hospital_id, summary_date)                   AS distinct_grain,
    SUM(total_tests)                                            AS total_tests,
    (SELECT COUNT(*) FROM {catalog}.silver.lab_results)         AS silver_lab_results,
    COUNT_IF(hospital_name IS NULL)                             AS missing_hospital_name,
    COUNT_IF(day_name IS NULL)                                  AS missing_calendar,
    COUNT_IF(prev_day_total_tests IS NULL)                      AS first_day_rows,
    COUNT(DISTINCT hospital_id)                                 AS hospitals
FROM {catalog}.gold.hospital_daily_summary
"""))

In [0]:
record_count = (spark.read.table("caresyncsydatabricks.gold.hospital_daily_summary").count())

dbutils.notebook.exit(str(record_count))